# CORUNA SCRAPPING

## Parseo de robots.txt

Ao comprobar se Wikipedia permitía descargar a páxina dos municipios da Coruña mediante `RobotFileParser`, obtíñase inicialmente o resultado `False`.

O problema estaba na forma de descargar o ficheiro `robots.txt`. Ao utilizar o método `rp.read()`, a petición realizábase directamente co axente de usuario predeterminado de Python, que podía ser rexeitado polo servidor de Wikipedia.

Para solucionalo, utilizouse a biblioteca `requests` para descargar o ficheiro `robots.txt`, especificando un `User-Agent` propio:

```python
respuesta = requests.get(
    url_robots,
    headers={"User-Agent": "curso-sbd-demo/1.0"}
)
```

Despois, o contido descargado pasouse a `RobotFileParser` mediante o método `parse()`:

```python
rp = RobotFileParser()
rp.parse(respuesta.text.splitlines())
```

Deste xeito, `requests` encárgase de descargar o ficheiro e `RobotFileParser` de interpretar as súas regras e comprobar se está permitido acceder á páxina mediante `can_fetch()`.

Con esta modificación, a comprobación realizouse correctamente.

In [ ]:
from urllib.robotparser import RobotFileParser
import requests

# Descargar robots.txt usando un User-Agent propio
url_robots = "https://en.wikipedia.org/robots.txt"

respuesta = requests.get(
    url_robots,
    headers={"User-Agent": "curso-sbd-demo/1.0"},
)

print("Estado HTTP:", respuesta.status_code)

# Leer las reglas con RobotFileParser
rp = RobotFileParser()
rp.parse(respuesta.text.splitlines()) # recibe la lista, la analiza y la guarda en memoria

# Comprobar la página
pagina = "https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a"

print("¿Se permite descargarla?:", rp.can_fetch("curso-sbd-demo/1.0", pagina)) # consulta las reglas que habiamos guardado en rp y da un veredicto

Estado HTTP: 200
¿Se permite descargarla?: True


O veredicto do Parser dinos que se nos permite descargar o contido da entrada da Wikipedia sobre os concellos da Coruña.

## Descarga

In [6]:
import time
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests
from bs4 import BeautifulSoup


Configuración inicial e directorios de traballo

In [7]:

BASE = "https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a"
CABECEIRAS = {"User-Agent": "curso-sbd-demo/1.0 (docente@exemplo.gal)"}
PAUSA = 1.0                      # segundos entre peticións

RAW = Path("../datos/raw/html")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

Descarga e comprobación da resposta HTTP

In [8]:
resposta = requests.get(BASE, headers=CABECEIRAS, timeout=10) # esperar 10 segundos como máximo
print("Código de estado:", resposta.status_code) # EL BASE devolve 200 se todo foi ben
print("Content-Type    :", resposta.headers.get("Content-Type")) # CABECEIRAS devolve text/html; charset=utf-8
print("Codificación    :", resposta.encoding)
print()
print(resposta.text[:600])

Código de estado: 200
Content-Type    : text/html; charset=UTF-8
Codificación    : UTF-8

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-available skin-thumbsize-clientpref-standard skin-un


- Código de estado: 200 (está todo OK)
- Content type: o contido descargado é HTML con codificación UTF-8

Utilizaremos BeautifulSoup para percorrer a estrutura HTML da resposta, extraer a táboa cos datos dos municipios e posteriormente transformala nun DataFrame de pandas.

## Extracción

BeautifulSoup converte o texto HTML unha árbore DOM (xerárquica) que podemos percorrer e na que podemos buscar. Por exemplo:

- `sopa.title.get_text()`: extrae o <title> do HTML.
- `sopa.find("h1").get_text()`: extrae o primeiro <h1>.

In [9]:
sopa = BeautifulSoup(resposta.content, "html.parser")

print(sopa.title.get_text())
print(sopa.find("h1").get_text())

List of municipalities in A Coruña - Wikipedia
List of municipalities in A Coruña


Pero queremos atopar a lista de datos dos concellos. Para elo, buscamos todos os obxectos "table" do HTML
e vemos qué clases CSS ten a táboa que nos interesa (neste caso a primeira das que aparece). 

In [13]:
for tabla in sopa.find_all("table")[:2]:
    print(tabla.get("class"))

['wikitable', 'sortable']
['nowraplinks', 'mw-collapsible', 'autocollapse', 'navbox-inner']


Buscamos a táboa que teña clase "wikitable". Facemos un print e confirmamos que é a lista dos concellos da Coruña.

In [15]:
tabla = sopa.find("table", class_="wikitable")

print(tabla.get_text()[:200])


Name
Population (2024)[20]

Abegondo
5,578

Ames
32,812

Aranga
1,799

Ares
6,156

Arteixo
34,038

Arzúa
5,887

A Baña
3,291

Bergondo
6,986

Betanzos
13,261

Boimorto
1,801

Boiro
19,018

Boqueixón



Finalmente construímos o DataFrame.

In [16]:
datos = []

for fila in tabla.find_all("tr")[1:]:
    celdas = fila.find_all("td")

    if len(celdas) >= 2:
        concello = celdas[0].get_text(" ", strip=True)
        poboacion = celdas[1].get_text(" ", strip=True)

        datos.append({
            "Concello": concello,
            "Poboacion": poboacion
        })

df = pd.DataFrame(datos)

display(df.head())

,Concello,Poboacion
0,Abegondo,"5,578"
1,Ames,"32,812"
2,Aranga,"1,799"
3,Ares,"6,156"
4,Arteixo,"34,038"


## Limpeza

Vemos que temos os datos de poboación aínda sn formato string. Antes de pasalos a enteiros, comprobaremos que todos os valores teñen o mesmo formato.

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Concello   93 non-null     str  
 1   Poboacion  93 non-null     str  
dtypes: str(2)
memory usage: 1.6 KB


Usando o método `str.contains` comprobamos que os valores utilizan a coma para as miles e ningún usa puntos. É o esperable tendo en 
conta que son os datos da Wikipedia en inglés. Polo tanto, podemos pasalos xa a enteiros.

In [19]:
# Comprobamos cuántos registros contienen comas (posibles separadores decimales).
print("Presenza de comas:")
print(df["Poboacion"].str.contains(",", na=False).value_counts())

# Comprobamos cuántos registros contienen puntos (posibles separadores de miles).
print("\nPresenza de puntos:")
print(df["Poboacion"].str.contains(r"\.", na=False).value_counts())

# Mostramos los valores que contienen comas para inspeccionar su formato.
print("\nValores que conteñen comas:")
print(df.loc[df["Poboacion"].str.contains(",", na=False), "Poboacion"])

Presenza de comas:
Poboacion
True    93
Name: count, dtype: int64

Presenza de puntos:
Poboacion
False    93
Name: count, dtype: int64

Valores que conteñen comas:
0      5,578
1     32,812
2      1,799
3      6,156
4     34,038
       ...  
88     4,942
89     1,252
90     1,229
91     6,808
92     4,261
Name: Poboacion, Length: 93, dtype: str


Pasámolos a enteiros e comprobamos con `.info()`.

In [21]:
df["Poboacion"] = (
    df["Poboacion"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Concello   93 non-null     str  
 1   Poboacion  93 non-null     int64
dtypes: int64(1), str(1)
memory usage: 1.6 KB


## Validación
### Comprobación do número de concellos e ningún repetido

Validamos os datos: precisamos saber cantos concellos temos e se son únicos. Nos seguintes prints, valéndonos do método `.nunique()` e de `.duplicated()` observamos que son todos os concellos únicos. 

In [23]:
print("Número de concellos:", len(df))
print("Número de concellos únicos:", df["Concello"].nunique())
print("Número de duplicados:", df["Concello"].duplicated().sum())

Número de concellos: 93
Número de concellos únicos: 93
Número de duplicados: 0


### Poboación total segundo o dataframe VS segundo a wikipedia

Mediante `sum()` obtemos un total de 1.128.320 habitantes na provincia da Coruña. Segundo a Wikipedia, na *List of municipalities in A Coruña*, recordemos a ligazón (https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a), a poboación total é de *1,128,320 inhabitants*. 

É coincidente, aínda que non é o mesmo dato que aparece na páxina principal adicada á provincia da Coruña, onde pon que ten *1,135,623* habitantes; probablemente sexan datos máis recentes.

In [24]:
total_df = df["Poboacion"].sum()

print("Poboación total segundo o DataFrame:", total_df)

Poboación total segundo o DataFrame: 1128320


## Intento do mesmo proceso con pandas

Intentamos ler o HTML usando pandas e chegar ao mesmo resultado que chegamos con BeautifulSoup

In [29]:
from io import StringIO

tablas = pd.read_html(StringIO(resposta.text))

print("Número de táboas atopadas:", len(tablas))

Número de táboas atopadas: 2


In [32]:
for i, t in enumerate(tablas):
    print(f"Táboa {i}: {t.shape}")
    print(t.columns.tolist())

Táboa 0: (93, 2)
['Name', 'Population (2024)[20]']
Táboa 1: (4, 2)
['vteMunicipalities of Spain', 'vteMunicipalities of Spain.1']


In [33]:
df_read_html = tablas[0]

print("Dimensión:", df_read_html.shape)
print("Columnas:", df_read_html.columns.tolist())
display(df_read_html.head())

Dimensión: (93, 2)
Columnas: ['Name', 'Population (2024)[20]']


,Name,Population (2024)[20]
0,Abegondo,5578
1,Ames,32812
2,Aranga,1799
3,Ares,6156
4,Arteixo,34038


In [34]:
df_read_html.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   Name                   93 non-null     str  
 1   Population (2024)[20]  93 non-null     int64
dtypes: int64(1), str(1)
memory usage: 1.6 KB


O proceso é similar salvo por unha pequena diferenza. `read_html` interpreta directamente os valores da columna *Population*, que en principio están en string, como valores que deben ser pasados a enteiros. Ademais, tamén interpreta correctamente que as comas representan un separador de miles. Esto é posible porque os valores veñen da Wikipedia en inglés, se estivésemos facendo a mesma tarefa con datos extraídos da Wikipedia en galego ou en español, daríanos erro.

Facemos a suma dos concellos e comprobamos que sí da o mesmo resultado.

In [35]:
df_read_html["Population (2024)[20]"].sum()


np.int64(1128320)

## Cuestións

1. Os 5 concellos máis poboados.

Número 1: A Coruña, con 249.261
Número 2: Santiago de Compostela, con 99.56
Número 3: Ferrol, con 64.218
Número 4: Narón, con 39.285
Número 5: Oleiros, con 38.333

In [57]:
df.iloc[df["Poboacion"].sort_values(ascending=False).head(5).index.tolist()]

,Concello,Poboacion
29,A Coruña,249261
77,Santiago de Compostela,99536
35,Ferrol,64218
53,Narón,39285
57,Oleiros,38333


2. Número de concellos con menos de 5.000 habitantes e porcentaxe sobre o total.

Hai 43 concellos con menos de 5000 habitantes.

In [67]:
len(df[df["Poboacion"]<5000])

43

A poboación destes concellos representa tan só un 10.6 % dos habitantes totais da provincia

In [ ]:
round(df[df["Poboacion"]<5000].loc[:, ["Poboacion"]].sum()/df["Poboacion"].sum()*100, 2)

Poboacion    10.6
dtype: float64

3. Porcentaxe de poboación que vive nos tres concellos máis poboados (A Coruña, Santiago e Ferrol). 

Un 36.6 % da poboación vive nos tres concellos máis poboados.

In [80]:
round(df.iloc[df["Poboacion"].sort_values(ascending=False).head(3).index.tolist()].loc[:, ["Poboacion"]].sum()/df["Poboacion"].sum()*100, 2)

Poboacion    36.6
dtype: float64

4. Cantos concellos fan falta, se se ordenan de máis a menos poboados, para reunir a metade da poboación

Con 8 concellos basta para reunir a metade da poboación.

In [84]:
concellos_descendentes = df["Poboacion"].sort_values(ascending=False)
metade = concellos_descendentes.sum()/2
count = 0
for i in concellos_descendentes:
    if metade > 0:
        metade -= i
        count += 1
    else:
        break
count

8

In [89]:
print(f'La metade da poboación é: {concellos_descendentes.sum()/2}')
print(f'Os 8 concellos co maior poboación suman: {df["Poboacion"].sort_values(ascending=False).head(8).sum()}')

La metade da poboación é: 564160.0
Os 8 concellos co maior poboación suman: 589078


4.7 Gardar
Garda o resultado limpo en datos/clean/, en CSV con codificación UTF-8.


## Gardar os datos en datos/clean

Gardamos os datos limpos nun.csv en clean.

In [92]:
from pathlib import Path

print(Path.cwd())
print(Path("../datos/clean/concellos.csv").resolve())

/home/estev/code/concellos_corunha/notebooks
/home/estev/code/concellos_corunha/datos/clean/concellos.csv
